# Observables and Shot Planning Fundamentals

This notebook demonstrates that Hoeffding shot planning works for **any** bounded observable, not just SWAP tests. We'll explore single-qubit Pauli observables (X, Y, Z) and show how to plan shots for each.

## Key Concepts

1. **Observables**: Physical quantities we want to measure (e.g., Pauli X, Y, Z)
2. **Eigenvalues**: Possible measurement outcomes for a single shot
3. **Bounded ranges**: All Pauli observables have eigenvalues ±1 → bounded in [-1, 1]
4. **Hoeffding planning**: Works for ANY bounded random variable

In [1]:
import math
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler import generate_preset_pass_manager

from qamp_shotplanner import (
    HoeffdingPlanner,
    pauli_x,
    pauli_y,
    pauli_z,
    single_qubit_observable,
)

## 1. Single-Qubit State Preparation

Let's prepare a single-qubit state using a Y-rotation:

$|\psi(\theta)\rangle = R_y(\theta)|0\rangle = \cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$

In [2]:
# Prepare a single-qubit state
theta = math.pi / 4  # 45 degrees

qc = QuantumCircuit(1)
qc.ry(theta, 0)

print(f"Prepared state: |ψ⟩ = Ry({theta:.4f})|0⟩")
print(f"Circuit: {qc.num_qubits} qubit(s)")

Prepared state: |ψ⟩ = Ry(0.7854)|0⟩
Circuit: 1 qubit(s)


## 2. The Three Pauli Observables

### Pauli Matrices
$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad
Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}, \quad
Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$$

### Key Properties
- **Eigenvalues**: All have eigenvalues ±1
- **Per-shot outcomes**: Each measurement yields +1 or -1
- **Bounded range**: Outcomes bounded in [-1, 1]
- **Expectation values**: $\langle X \rangle, \langle Y \rangle, \langle Z \rangle \in [-1, 1]$

In [3]:
# Create observables for our single-qubit system
obs_X = pauli_x(qubit=0, num_qubits=1)
obs_Y = pauli_y(qubit=0, num_qubits=1)
obs_Z = pauli_z(qubit=0, num_qubits=1)

print("Pauli Observables:")
print(f"X observable: {obs_X}")
print(f"Y observable: {obs_Y}")
print(f"Z observable: {obs_Z}")

Pauli Observables:
X observable: SparsePauliOp(['X'],
              coeffs=[1.+0.j])
Y observable: SparsePauliOp(['Y'],
              coeffs=[1.+0.j])
Z observable: SparsePauliOp(['Z'],
              coeffs=[1.+0.j])


## 3. Shot Planning for Different Observables

Since all Pauli observables are bounded in [-1, 1], the shot planning is **identical** for all of them!

In [4]:
# Shot planning parameters
epsilon = 0.02  # tolerance on expectation value error
delta = 0.01    # failure probability

# All Pauli observables have the same bounds [-1, 1]
planner = HoeffdingPlanner(
    epsilon_stat=epsilon,
    delta=delta,
    a=-1.0,  # Lower bound
    b=+1.0,  # Upper bound
)

shots = planner.planned_shots()
print(f"Planned shots for ANY Pauli observable: {shots}")
print(f"Guarantee: |Ê - E| ≤ {epsilon} with probability ≥ {1-delta:.0%}")

Planned shots for ANY Pauli observable: 26492
Guarantee: |Ê - E| ≤ 0.02 with probability ≥ 99%


## 4. Measuring Each Observable

Let's measure X, Y, and Z on the same state and compare with theoretical predictions.

In [5]:
# Helper function to run estimator
def measure_observable(qc, observable, shots):
    backend = AerSimulator()
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    qc_isa = pm.run(qc)
    
    options = {"run_options": {"shots": shots}, "backend_options": {"seed_simulator": 42}}
    estimator = AerEstimator(options=options)
    
    job = estimator.run([(qc_isa, observable, [])])
    return float(job.result()[0].data.evs)

# Measure all three observables
E_X = measure_observable(qc, obs_X, shots)
E_Y = measure_observable(qc, obs_Y, shots)
E_Z = measure_observable(qc, obs_Z, shots)

print("Measured Expectation Values:")
print(f"E[X] = {E_X:.6f}")
print(f"E[Y] = {E_Y:.6f}")
print(f"E[Z] = {E_Z:.6f}")

Measured Expectation Values:
E[X] = 0.707107
E[Y] = 0.000000
E[Z] = 0.707107


## 5. Theoretical Predictions

For state $|\psi\rangle = R_y(\theta)|0\rangle$:

$$\langle Z \rangle = \cos\theta$$
$$\langle X \rangle = \sin\theta$$
$$\langle Y \rangle = 0$$

In [6]:
# Theoretical predictions
E_Z_theory = math.cos(theta)
E_X_theory = math.sin(theta)
E_Y_theory = 0.0

print("Theoretical vs. Measured:")
print(f"E[Z]: theory = {E_Z_theory:.6f}, measured = {E_Z:.6f}, error = {abs(E_Z - E_Z_theory):.6f}")
print(f"E[X]: theory = {E_X_theory:.6f}, measured = {E_X:.6f}, error = {abs(E_X - E_X_theory):.6f}")
print(f"E[Y]: theory = {E_Y_theory:.6f}, measured = {E_Y:.6f}, error = {abs(E_Y - E_Y_theory):.6f}")

print(f"\nAll errors should be ≤ {epsilon} (target tolerance)")

Theoretical vs. Measured:
E[Z]: theory = 0.707107, measured = 0.707107, error = 0.000000
E[X]: theory = 0.707107, measured = 0.707107, error = 0.000000
E[Y]: theory = 0.000000, measured = 0.000000, error = 0.000000

All errors should be ≤ 0.02 (target tolerance)


## 6. Exploring Different States

Let's see how the observables change for different rotation angles.

In [7]:
# Test different rotation angles
thetas = [0, math.pi/6, math.pi/4, math.pi/3, math.pi/2]

print("θ\tE[Z]\tE[X]\tE[Y]\tTheory Z\tTheory X\tTheory Y")
print("-" * 70)

for theta_test in thetas:
    qc_test = QuantumCircuit(1)
    qc_test.ry(theta_test, 0)
    
    E_Z_test = measure_observable(qc_test, obs_Z, shots)
    E_X_test = measure_observable(qc_test, obs_X, shots)
    E_Y_test = measure_observable(qc_test, obs_Y, shots)
    
    E_Z_theory_test = math.cos(theta_test)
    E_X_theory_test = math.sin(theta_test)
    E_Y_theory_test = 0.0
    
    print(f"{theta_test:.4f}\t{E_Z_test:.3f}\t{E_X_test:.3f}\t{E_Y_test:.3f}\t{E_Z_theory_test:.3f}\t{E_X_theory_test:.3f}\t{E_Y_theory_test:.3f}")

θ	E[Z]	E[X]	E[Y]	Theory Z	Theory X	Theory Y
----------------------------------------------------------------------
0.0000	1.000	0.000	0.000	1.000	0.000	0.000
0.5236	0.866	0.500	0.000	0.866	0.500	0.000
0.7854	0.707	0.707	0.000	0.707	0.707	0.000
1.0472	0.500	0.866	0.000	0.500	0.866	0.000
1.5708	0.000	1.000	0.000	0.000	1.000	0.000


## 7. Key Takeaways

### Universality of Hoeffding Planning
1. **All Pauli observables are bounded in [-1, 1]**
2. **Same shot planning applies to all**: 26,492 shots for ε=0.02, δ=0.01
3. **No observable-specific tuning needed**: Just use the bounds

### Practical Implications
- **X, Y, Z measurements**: Same shot planning
- **Single-qubit vs multi-qubit**: Shot planning depends only on bounds
- **Different observables**: Just change the observable, not the planning

### For Next Notebook
- Multi-qubit correlations (XX, YY, ZZ)
- Entanglement detection
- Bell state analysis